# Adrianic v4.0 — Semantic Bootstrap / Transfer Lab

This is the first explicit semantic-transfer harness.

The core model is deliberately ordinary: a small word-level recurrent state-space predictor trained only on **next-token prediction**.

The experimental variable is the training/memory framework around it:

- provisional HOLD,
- INCLUDE / EXCLUDE / `VALUE_UNRESOLVED`,
- transparent evidence-source dependency discounting,
- selective replay,
- parity-protected replay storage,
- and sequential corpus transfer.

No semantic category, fact label, religious interpretation, literary interpretation, or answer key is supplied to model training. Ground-truth fact labels are evaluation-only.

## Corpora

**Local smoke mode:** a controlled STEM/reality corpus plus an unrelated synthetic literary distractor.

**Full Colab mode:** the controlled STEM/reality corpus plus Tiny Shakespeare and the public-domain King James Bible. The notebook downloads those corpora at runtime.

The natural-corpus tests are intentionally not treated as proof of human-like semantics. We measure retention, predictive structure, and answer ranking first.

## Colab launch

Upload this notebook or the Python script to Colab, then run the code cell below.

The runtime root is detected automatically (`/content` in Colab, `/mnt/data` in the OpenAI sandbox).

For a first natural-corpus pass:

```python
!python /content/Adrianic_v4_0_Semantic_Bootstrap_Transfer_DIRECT_VERIFIED.py --full --seeds 3
```

If you uploaded the script under a different filename/path, change that path accordingly.


## Suggested Colab run

1. Use a GPU runtime if available.
2. Run the setup/code cell.
3. Run:

```python
!python /content/Adrianic_v4_0_Semantic_Bootstrap_Transfer_DIRECT_VERIFIED.py --full --seeds 5
```

For a first pass, `--seeds 3` is reasonable. Increase after the pipeline is behaving.

In [ ]:

from __future__ import annotations
from pathlib import Path
import argparse, json, math, random, re, hashlib, zipfile, shutil, urllib.request
from collections import Counter
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

# Portable runtime root:
# - Colab normally uses /content
# - OpenAI sandbox uses /mnt/data
# - otherwise fall back to the current working directory
if Path("/content").exists():
    ROOT = Path("/content")
elif Path("/mnt/data").exists():
    ROOT = Path("/mnt/data")
else:
    ROOT = Path.cwd()

OUT = ROOT/"adrianic_v40_semantic_bootstrap_results"
OUT.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Reproducibility
# -----------------------------
def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

# -----------------------------
# Corpus construction
# -----------------------------
FACTS = [
    # math
    ("two plus two equals", "four", "math"),
    ("three plus five equals", "eight", "math"),
    ("seven minus three equals", "four", "math"),
    ("six times seven equals", "fortytwo", "math"),
    ("the square root of nine is", "three", "math"),
    ("a triangle has", "three", "math"),
    ("a square has", "four", "math"),
    ("an angle measured at ninety degrees is", "right", "math"),
    ("the ratio of circumference to diameter is called", "pi", "math"),
    ("the additive identity is", "zero", "math"),
    # physics
    ("the si unit of force is the", "newton", "physics"),
    ("the si unit of energy is the", "joule", "physics"),
    ("the si unit of power is the", "watt", "physics"),
    ("the si unit of electric current is the", "ampere", "physics"),
    ("the speed of light in vacuum is approximately three hundred million meters per", "second", "physics"),
    ("mass times acceleration gives", "force", "physics"),
    ("energy associated with motion is", "kinetic", "physics"),
    ("energy stored by position can be", "potential", "physics"),
    ("the force attracting masses is", "gravity", "physics"),
    ("electric charge is measured in", "coulombs", "physics"),
    # chemistry
    ("the chemical symbol for hydrogen is", "h", "chemistry"),
    ("the chemical symbol for oxygen is", "o", "chemistry"),
    ("the chemical symbol for carbon is", "c", "chemistry"),
    ("water contains hydrogen and", "oxygen", "chemistry"),
    ("common table salt contains sodium and", "chlorine", "chemistry"),
    ("an atom with six protons is", "carbon", "chemistry"),
    ("an atom with eight protons is", "oxygen", "chemistry"),
    ("a substance with ph below seven is", "acidic", "chemistry"),
    ("a substance with ph above seven is", "basic", "chemistry"),
    ("the center of an atom is the", "nucleus", "chemistry"),
    # world / astronomy
    ("the planet closest to the sun is", "mercury", "world"),
    ("earth orbits the", "sun", "world"),
    ("the moon orbits the", "earth", "world"),
    ("the largest planet in the solar system is", "jupiter", "world"),
    ("liquid water freezes near zero degrees", "celsius", "world"),
    ("liquid water boils near one hundred degrees", "celsius", "world"),
    ("plants use light in", "photosynthesis", "world"),
    ("humans breathe in", "oxygen", "world"),
    ("the gas most abundant in earth atmosphere is", "nitrogen", "world"),
    ("solid liquid and gas are states of", "matter", "world"),
]

TEMPLATES = [
    "{p} {a}.",
    "fact: {p} {a}.",
    "in this lesson, {p} {a}.",
    "remember that {p} {a}.",
]

def make_stem_corpus(seed=0, repeats=5, junk=120):
    rng=random.Random(seed)
    sentences=[]
    labels=[]
    for prompt,ans,domain in FACTS:
        for r in range(repeats):
            t=TEMPLATES[r % len(TEMPLATES)]
            sentences.append(t.format(p=prompt,a=ans))
            labels.append(("fact",domain,prompt,ans))
    # One-off junk uses essentially unique alphabetic tokens. This makes
    # recurrence a legitimate unsupervised signal rather than a hidden label.
    alphabet="abcdefghijklmnopqrstuvwxyz"
    def code(n):
        out=""
        x=n+1
        while x:
            x,r=divmod(x-1,26)
            out=alphabet[r]+out
        return "noise"+out
    cursor=0
    for j in range(junk):
        words=[code(cursor+k) for k in range(7)]
        cursor+=7
        rng.shuffle(words)
        sentences.append(" ".join(words)+".")
        labels.append(("junk","junk","", ""))
    order=list(range(len(sentences))); rng.shuffle(order)
    return [sentences[i] for i in order], [labels[i] for i in order]

def make_proxy_distractor(seed=0, n=500):
    rng=random.Random(seed)
    names=["arden","cassian","lyra","orion","mara","selene","tavian","viola"]
    verbs=["whispers","wanders","argues","laughs","waits","returns","dreams","speaks"]
    objs=["beneath the moon","beside the gate","through the hall","before the crowd",
          "near the river","under the tower","after the feast","beyond the garden"]
    return [f"{rng.choice(names)} {rng.choice(verbs)} {rng.choice(objs)}." for _ in range(n)]

def download_public_corpora():
    """
    Intended for Colab/full mode. Public-domain / open benchmark sources.
    """
    paths={}
    urls={
        "shakespeare":"https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt",
        "bible":"https://www.gutenberg.org/cache/epub/10900/pg10900.txt",
    }
    for name,url in urls.items():
        p=ROOT/f"{name}_corpus.txt"
        if not p.exists():
            urllib.request.urlretrieve(url,p)
        paths[name]=p
    return paths

# -----------------------------
# Tokenizer / vocabulary
# -----------------------------
TOKEN_RE=re.compile(r"[a-zA-Z]+|[0-9]+|[^\w\s]")

def tokenize(text):
    return TOKEN_RE.findall(text.lower())

class Vocab:
    def __init__(self,texts,max_vocab=7000):
        c=Counter()
        for t in texts: c.update(tokenize(t))
        self.itos=["<pad>","<unk>"]+[w for w,_ in c.most_common(max_vocab-2)]
        self.stoi={w:i for i,w in enumerate(self.itos)}
    def encode(self,text):
        return [self.stoi.get(w,1) for w in tokenize(text)]
    def decode(self,ids):
        return " ".join(self.itos[i] for i in ids)

# -----------------------------
# Tiny state-space language model
# -----------------------------
class TinySSM(nn.Module):
    """
    Small word-level recurrent state-space core.
    Training signal is only next-token prediction.
    No semantic category/fact labels enter optimization.
    """
    def __init__(self,vocab_size,emb=48,hid=72):
        super().__init__()
        self.emb=nn.Embedding(vocab_size,emb)
        self.rnn=nn.RNN(emb,hid,nonlinearity="tanh",batch_first=True)
        self.out=nn.Linear(hid,vocab_size)
    def forward(self,x,h=None):
        e=self.emb(x)
        z,h=self.rnn(e,h)
        return self.out(z),h

def seq_loss(model,ids,device="cpu"):
    if len(ids)<2:return float("nan")
    x=torch.tensor(ids[:-1],dtype=torch.long,device=device)[None,:]
    y=torch.tensor(ids[1:],dtype=torch.long,device=device)[None,:]
    with torch.no_grad():
        logits,_=model(x)
        return float(nn.functional.cross_entropy(logits.reshape(-1,logits.size(-1)),y.reshape(-1)).item())

def train_sentence(model,opt,ids,device="cpu"):
    if len(ids)<2:return float("nan")
    x=torch.tensor(ids[:-1],dtype=torch.long,device=device)[None,:]
    y=torch.tensor(ids[1:],dtype=torch.long,device=device)[None,:]
    opt.zero_grad(set_to_none=True)
    logits,_=model(x)
    loss=nn.functional.cross_entropy(logits.reshape(-1,logits.size(-1)),y.reshape(-1))
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
    opt.step()
    return float(loss.item())

# -----------------------------
# Deterministic selective replay / evidence graph
# -----------------------------
STOP=set("the a an is are in of to and this that with for near under before after through beyond".split())

def chunk_features(sentences,vocab,model):
    tok_counts=Counter()
    chunk_tokens=[]
    for s in sentences:
        toks=[w for w in tokenize(s) if w not in STOP and w.isalpha()]
        chunk_tokens.append(toks); tok_counts.update(set(toks))

    rows=[]
    for i,(s,toks) in enumerate(zip(sentences,chunk_tokens)):
        ids=vocab.encode(s)
        recurrence=np.mean([math.log1p(tok_counts[w]) for w in toks]) if toks else 0.0
        cross_context=np.mean([tok_counts[w]>=3 for w in toks]) if toks else 0.0
        predictability=-seq_loss(model,ids)
        rows.append({"idx":i,"recurrence":recurrence,
                     "cross_context":cross_context,
                     "predictability":predictability})
    return pd.DataFrame(rows)

def evidence_graph_scores(df):
    """
    Transparent graph over evidence sources.
    Correlated signals discount each other; no semantic labels are used.

    Constant evidence channels are explicitly given zero weight instead of
    allowing a NaN correlation matrix to manufacture VALUE_UNRESOLVED.
    """
    cols=["recurrence","cross_context","predictability"]
    X=df[cols].to_numpy(float)
    mu=X.mean(0); sd=X.std(0)
    active=sd>1e-8
    Z=np.zeros_like(X)
    Z[:,active]=(X[:,active]-mu[active])/(sd[active]+1e-9)

    C=np.zeros((len(cols),len(cols)),float)
    idx=np.where(active)[0]
    if len(idx)>1:
        Ca=np.abs(np.corrcoef(Z[:,idx],rowvar=False))
        Ca=np.nan_to_num(Ca,nan=0.0,posinf=0.0,neginf=0.0)
        np.fill_diagonal(Ca,0.0)
        for a_i,i in enumerate(idx):
            for a_j,j in enumerate(idx):
                C[i,j]=Ca[a_i,a_j]

    rho=np.where(active,0.70,0.0)
    a=rho**3
    if a.sum()<=1e-12:
        return np.zeros(len(df)),np.zeros(len(cols)),C,0.0
    abar=a/(a.sum()+1e-12)
    redundancy=C@abar
    w=a/(1+1.5*redundancy)
    w=w/(w.sum()+1e-12)

    score=Z@w
    confidence=1.0/(1.0+np.mean(C))
    return score,w,C,confidence

class ParityProtectedMemory:
    """
    Two-copy replay store with two deterministic checksums.
    Parity is error management only; it never supplies semantic value.
    """
    PRIME=1000003
    def __init__(self):
        self.items=[]
    @classmethod
    def checksums(cls,ids):
        ids=np.asarray(ids,dtype=np.int64)
        p=int(ids.sum()%cls.PRIME)
        q=int(((np.arange(len(ids))+1)*ids).sum()%cls.PRIME)
        return p,q
    def add(self,ids,meta):
        p,q=self.checksums(ids)
        self.items.append({"primary":list(ids),"backup":list(ids),"p":p,"q":q,"meta":meta})
    def corrupt(self,prob,seed):
        rng=random.Random(seed)
        for item in self.items:
            if rng.random()<prob and item["primary"]:
                j=rng.randrange(len(item["primary"]))
                item["primary"][j]=(item["primary"][j]+rng.randint(1,17))%100000
    def retrieve(self):
        repaired=0; lost=0; out=[]
        for item in self.items:
            p,q=self.checksums(item["primary"])
            if (p,q)==(item["p"],item["q"]):
                out.append(item["primary"]); continue
            # primary failed; verify independent backup
            p2,q2=self.checksums(item["backup"])
            if (p2,q2)==(item["p"],item["q"]):
                out.append(item["backup"]); repaired+=1
            else:
                lost+=1
        return out,repaired,lost

def build_memory(sentences,labels,vocab,model,keep_frac=.28):
    feat=chunk_features(sentences,vocab,model)
    score,w,C,conf=evidence_graph_scores(feat)
    feat["score"]=score

    # Provenance/dependency-aware diversity:
    # repeated lexical variants are not counted as fully independent evidence.
    # Greedy selection discounts candidates that are near-duplicates of already
    # selected chunks. This uses token overlap only, never semantic labels.
    token_sets=[]
    for s in sentences:
        token_sets.append(set(w for w in tokenize(s) if w not in STOP and w.isalpha()))

    k=max(1,int(round(keep_frac*len(sentences))))
    selected=[]
    remaining=set(range(len(sentences)))
    diversity_lambda=.85
    while remaining and len(selected)<k:
        best=None; best_val=-1e30
        for i in remaining:
            if selected and token_sets[i]:
                mx=max(
                    len(token_sets[i]&token_sets[j])/(len(token_sets[i]|token_sets[j])+1e-12)
                    for j in selected if token_sets[j]
                )
            else:
                mx=0.0
            val=float(score[i])-diversity_lambda*mx
            if val>best_val:
                best_val=val; best=i
        selected.append(best); remaining.remove(best)

    selected=set(selected)
    lo=np.quantile(score,.25)

    mem=ParityProtectedMemory()
    decisions=[]
    for i,s in enumerate(sentences):
        if i in selected:
            decision="INCLUDE"
            mem.add(vocab.encode(s),{"source_idx":i})
        elif score[i]<=lo:
            decision="EXCLUDE"
        else:
            decision="VALUE_UNRESOLVED"
        decisions.append(decision)
    feat["decision"]=decisions

    # Evaluation only.
    eval_kind=[labels[i][0] for i in feat["idx"]]
    feat["eval_kind"]=eval_kind
    return mem,feat,{
        "weights":w.tolist(),"dependency":C.tolist(),"confidence":float(conf),
        "diversity_lambda":diversity_lambda
    }

# -----------------------------
# QA evaluation
# -----------------------------
def answer_rank(model,vocab,prompt,answer,device="cpu"):
    ids=vocab.encode(prompt)
    if not ids:return None
    x=torch.tensor(ids,dtype=torch.long,device=device)[None,:]
    with torch.no_grad():
        logits,_=model(x)
        v=logits[0,-1]
        ans=vocab.stoi.get(answer,1)
        rank=int((v>v[ans]).sum().item())+1
        return rank

def qa_metrics(model,vocab):
    ranks=[]
    domain_rows=[]
    for prompt,ans,domain in FACTS:
        r=answer_rank(model,vocab,prompt,ans)
        ranks.append(r)
        domain_rows.append((domain,r))
    top1=np.mean([r==1 for r in ranks])
    top5=np.mean([r<=5 for r in ranks])
    mrr=np.mean([1/r for r in ranks])
    return {"top1":float(top1),"top5":float(top5),"mrr":float(mrr),
            "median_rank":float(np.median(ranks))},domain_rows

# -----------------------------
# Training experiment
# -----------------------------
def run_one(seed=0,mode="smoke",device="cpu"):
    seed_all(seed)
    stem_sents,stem_labels=make_stem_corpus(seed,repeats=5,junk=120)

    if mode=="full":
        paths=download_public_corpora()
        sh=paths["shakespeare"].read_text(errors="ignore")
        bi=paths["bible"].read_text(errors="ignore")
        # Keep runtime bounded but genuinely natural.
        distractors={
            "shakespeare":re.split(r"(?<=[.!?])\s+",sh[:700000]),
            "bible":re.split(r"(?<=[.!?])\s+",bi[:1000000]),
            "combined":re.split(r"(?<=[.!?])\s+",sh[:350000]+bi[:500000]),
        }
        # Clean empties and cap sentence length.
        for k,v in list(distractors.items()):
            distractors[k]=[s for s in v if 4<=len(tokenize(s))<=80][:2500]
    else:
        distractors={"proxy":make_proxy_distractor(seed,650)}

    rows=[]
    memory_rows=[]
    graph_info={}

    for distractor_name,distractor in distractors.items():
        # Shared vocabulary for fair variants.
        vocab=Vocab(stem_sents+distractor,max_vocab=7000)

        for variant in ["baseline","random_replay","framework","framework_no_parity"]:
            seed_all(seed)
            model=TinySSM(len(vocab.itos)).to(device)
            opt=torch.optim.Adam(model.parameters(),lr=0.006)

            # Stage A: reality/STEM stream
            order=list(range(len(stem_sents)))
            for epoch in range(2 if mode=="smoke" else 3):
                random.Random(seed+epoch).shuffle(order)
                for i in order:
                    train_sentence(model,opt,vocab.encode(stem_sents[i]),device)

            qa_pre,_=qa_metrics(model,vocab)

            # Build memory only after Stage A; no semantic labels used by controller.
            mem,feat,ginfo=build_memory(stem_sents,stem_labels,vocab,model,keep_frac=.28)
            graph_info[f"{distractor_name}_seed{seed}"]=ginfo

            if variant=="random_replay":
                n=len(mem.items)
                rng=random.Random(seed+999)
                picks=rng.sample(range(len(stem_sents)),n)
                mem=ParityProtectedMemory()
                for i in picks:
                    mem.add(vocab.encode(stem_sents[i]),{"source_idx":i})

            # Evaluate memory selectivity only, not used for training decisions.
            src=[it["meta"]["source_idx"] for it in mem.items]
            fact_precision=float(np.mean([stem_labels[i][0]=="fact" for i in src])) if src else 0.0
            covered=set()
            for i in src:
                if stem_labels[i][0]=="fact":
                    covered.add((stem_labels[i][2],stem_labels[i][3]))
            fact_coverage=float(len(covered)/len(FACTS))

            # Fault injection: framework gets parity; no-parity has backups destroyed too.
            mem.corrupt(.18,seed+12345)
            if variant=="framework_no_parity":
                for item in mem.items:
                    item["backup"]=list(item["primary"])  # corruption copied into backup
            replay,repaired,lost=mem.retrieve()

            # Stage B: unrelated corpus + periodic replay
            order2=list(range(len(distractor)))
            for epoch in range(2 if mode=="smoke" else 2):
                random.Random(seed+100+epoch).shuffle(order2)
                for step,i in enumerate(order2):
                    train_sentence(model,opt,vocab.encode(distractor[i]),device)
                    if variant!="baseline" and step%6==0 and replay:
                        rid=replay[(step//10)%len(replay)]
                        train_sentence(model,opt,rid,device)

            qa_post,_=qa_metrics(model,vocab)

            # Adaptation metric: heldout distractor loss
            held=distractor[-min(80,len(distractor)):]
            dloss=float(np.mean([seq_loss(model,vocab.encode(s),device) for s in held if len(vocab.encode(s))>1]))

            rows.append({
                "seed":seed,"distractor":distractor_name,"variant":variant,
                "qa_pre_top1":qa_pre["top1"],"qa_post_top1":qa_post["top1"],
                "qa_pre_top5":qa_pre["top5"],"qa_post_top5":qa_post["top5"],
                "qa_pre_mrr":qa_pre["mrr"],"qa_post_mrr":qa_post["mrr"],
                "qa_post_median_rank":qa_post["median_rank"],
                "forgetting_mrr":qa_pre["mrr"]-qa_post["mrr"],
                "distractor_loss":dloss,
                "memory_items":len(mem.items),
                "memory_fact_precision_eval_only":fact_precision,
                "memory_fact_coverage_eval_only":fact_coverage,
                "parity_repaired":repaired,"parity_lost":lost,
            })
            if variant=="framework":
                f2=feat.copy()
                f2["seed"]=seed;f2["distractor"]=distractor_name
                memory_rows.append(f2)

    return pd.DataFrame(rows), (pd.concat(memory_rows,ignore_index=True) if memory_rows else pd.DataFrame()), graph_info

def aggregate(df):
    return df.groupby(["distractor","variant"])[
        ["qa_pre_top1","qa_post_top1","qa_pre_mrr","qa_post_mrr",
         "forgetting_mrr","distractor_loss","memory_fact_precision_eval_only",
         "memory_fact_coverage_eval_only","parity_repaired","parity_lost"]
    ].agg(["mean","median","std","count"])

def main():
    ap=argparse.ArgumentParser()
    ap.add_argument("--full",action="store_true")
    ap.add_argument("--seeds",type=int,default=None)
    args=ap.parse_args()
    mode="full" if args.full else "smoke"
    seeds=args.seeds if args.seeds is not None else (3 if mode=="smoke" else 5)
    device="cuda" if torch.cuda.is_available() and mode=="full" else "cpu"

    allr=[]; allm=[]; graphs={}
    for s in range(seeds):
        r,m,g=run_one(s,mode,device)
        allr.append(r);allm.append(m);graphs.update(g)
        print("finished seed",s)

    df=pd.concat(allr,ignore_index=True)
    memdf=pd.concat(allm,ignore_index=True) if allm else pd.DataFrame()
    summary=aggregate(df)

    df.to_csv(OUT/f"{mode}_runs.csv",index=False)
    memdf.to_csv(OUT/f"{mode}_memory_decisions.csv",index=False)
    summary.to_csv(OUT/f"{mode}_summary.csv")
    (OUT/f"{mode}_evidence_graph.json").write_text(json.dumps(graphs,indent=2))

    # Primary controlled checks
    q=df[df["distractor"]==("proxy" if mode=="smoke" else df["distractor"].iloc[0])]
    med=q.groupby("variant").median(numeric_only=True)
    checks={}
    if "framework" in med.index and "baseline" in med.index:
        checks["framework_retains_more_STEM_MRR_than_baseline"] = bool(
            med.loc["framework","qa_post_mrr"] > med.loc["baseline","qa_post_mrr"]
        )
        checks["framework_forgets_less_than_baseline"] = bool(
            med.loc["framework","forgetting_mrr"] < med.loc["baseline","forgetting_mrr"]
        )
    if "framework" in med.index and "random_replay" in med.index:
        checks["selective_framework_beats_random_replay_on_post_MRR"] = bool(
            med.loc["framework","qa_post_mrr"] > med.loc["random_replay","qa_post_mrr"]
        )
    if "framework" in med.index and "framework_no_parity" in med.index:
        checks["parity_protects_replay_under_memory_corruption"] = bool(
            med.loc["framework","qa_post_mrr"] >= med.loc["framework_no_parity","qa_post_mrr"]
        )

    report = f"""Adrianic v4.0 — Semantic Bootstrap / Transfer Harness

Mode
====
{mode}

Purpose
=======
Test whether the deterministic developmental-memory framework can be transplanted
onto a generic word-level recurrent state-space language model.

The model receives NO semantic labels during training. Its only model-learning
objective is next-token prediction.

The framework receives NO domain/fact labels. Candidate replay chunks are judged
from recurrence, cross-context persistence, and current predictability. Those
signals are dependency-discounted by a transparent evidence graph. High-evidence
chunks are INCLUDE, low-evidence chunks EXCLUDE, and the middle remains
VALUE_UNRESOLVED/HOLD.

Semantic fact/domain labels exist only for evaluation.

Controlled STEM/reality corpus
==============================
The included controlled corpus covers arithmetic, geometry, physics, chemistry,
astronomy, units, and ordinary physical facts. It also contains one-off junk
sentences so selective consolidation can be measured without telling the
controller what is factual.

Variants
========
baseline          : no replay memory
random_replay     : same replay capacity, random chunks
framework         : evidence-graph selective replay + parity-protected storage
framework_no_parity: same selective replay, but corrupted backup is not recoverable

Summary
=======
{summary.to_string()}

Checks
======
{json.dumps(checks,indent=2)}

Interpretation boundary
=======================
Passing this experiment would NOT establish human-like semantic understanding.
It would show a narrower result: a framework trained only through predictive
consequences can preserve and reuse text-encoded factual relationships better
than matched controls under sequential interference.

The full/Colab mode additionally downloads Tiny Shakespeare and the public-domain
King James Bible, then repeats the sequential-transfer test with natural corpora.
That larger natural-language sweep is intentionally separate from this local
mechanics smoke test.
"""
    (OUT/f"{mode}_REPORT.txt").write_text(report)
    (OUT/f"{mode}_checks.json").write_text(json.dumps(checks,indent=2))
    print(report)

if __name__=="__main__":
    main()
